# MindStride — EEG Motor Imagery Classification

Interactive notebook importing from the modular `src` package.
All experiments from the original monolithic notebook are available here,
but the logic lives in reusable Python modules.

In [ ]:
# Install (run once)
# !pip install -e ".[dev]"

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from src.config import load_config
from src.data import (
    download_dataset, load_raw_subjects,
    epoch_subjects, epoch_with_params,
    EEGDataset, subject_split, make_dataloaders,
)
from src.models import EEGNet
from src.engine import train, eval_step, cross_validate_subjects, cv_for_preprocessing
from src.pipelines import (
    MultiClassCSP, run_csp_ml_grid,
    compute_mu_power, find_best_mu_threshold, two_stage_predict,
    run_eegnet_grid, run_preprocessing_grid, run_joint_grid,
    butter_bandpass_filter,
)
from src.utils import (
    set_seeds, get_device,
    plot_training_curves, plot_confusion_matrix, plot_model_comparison,
    print_evaluation, save_model, predict_with_model,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
cfg = load_config()
set_seeds(cfg["seed"])
device = get_device()
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Data Loading

In [ ]:
subjects_data = download_dataset(
    cfg["data"]["dataset"],
    cfg["data"]["desired_runs"],
)

raw_data = load_raw_subjects(
    subjects_data,
    sfreq=cfg["data"]["sfreq"],
    n_subjects=cfg["data"].get("n_subjects"),
    cache_dir=cfg["data"].get("cache_dir"),
)

## 2. EDA on a Single Subject

In [ ]:
import mne

raw_obj = raw_data["001"]
print(raw_obj)
print(f"Channels: {len(raw_obj.ch_names)}, Sfreq: {raw_obj.info['sfreq']} Hz")

raw_copy = raw_obj.copy().filter(7., 30., fir_design="firwin", skip_by_annotation="edge")
sfreq = raw_copy.info["sfreq"]
data, times = raw_copy[:-1, int(sfreq * 1):int(sfreq * 60)]

fig = plt.subplots(figsize=(10, 8))
plt.plot(times, data.T)
plt.xlabel("Seconds"); plt.ylabel("uV")
plt.title("Channels 1-64 (filtered 7-30 Hz)")
plt.show()

raw_copy.compute_psd(fmax=50).plot()
plt.show()

## 3. Preprocessing & Epoching (64ch, 3-class)

In [ ]:
X_all, y_all, subjects_all, skipped = epoch_subjects(
    raw_data,
    event_id={"rest": 1, "left_hand": 2, "right_hand": 3},
    channels=None,
    low_freq=7.0, high_freq=30.0,
    tmin=0.0, tmax=4.0,
    label_offset=1,
    seed=cfg["seed"],
)

## 4. Subject-based Split

In [ ]:
split = subject_split(X_all, y_all, subjects_all, seed=cfg["seed"])
loaders = make_dataloaders(split, batch_size=cfg["dataloader"]["batch_size"])
print(f"Train: {len(split['X_train'])} | Val: {len(split['X_val'])} | Test: {len(split['X_test'])}")

## 5. EEGNet Single Run (64ch, 3-class)

In [ ]:
eeg_net = EEGNet(chans=64, classes=3, time_points=X_all.shape[2]).to(device)

weights = compute_class_weight("balanced", classes=np.unique(split["y_train"]), y=split["y_train"])
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(device))
optimizer = torch.optim.Adam(eeg_net.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

results, best_val = train(
    model=eeg_net,
    train_dataloader=loaders["train_loader"],
    val_dataloader=loaders["val_loader"],
    test_dataloader=loaders["test_loader"],
    optimizer=optimizer, loss_fn=loss_fn,
    scheduler=scheduler, device=device, epochs=50,
)

plot_training_curves(results, "EEGNet 64ch 3-class")
preds, labels = predict_with_model(eeg_net, loaders["test_loader"], device)
print_evaluation(labels, preds, ["rest", "left_hand", "right_hand"], "Test")
plot_confusion_matrix(labels, preds, ["rest", "left_hand", "right_hand"])

## 6. Cross-Validation (64ch)

In [ ]:
cv_mask = split["train_mask"] | split["val_mask"]
fold_results, mean_acc, std_acc = cross_validate_subjects(
    X_all[cv_mask], y_all[cv_mask], subjects_all[cv_mask],
    n_splits=5, epochs=50, lr=0.001, chans=64, classes=3, device=device,
)

## 7. EEGNet Grid Search (64ch)

In [ ]:
import itertools

param_grid = cfg["eegnet_grid"]
keys = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"Total combinations: {len(combos)}")

grid_results = []
for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    f2 = params["f1"] * params["d"]
    print(f"\nCombo {i+1}/{len(combos)}: {params}, f2={f2}")
    _, mean_acc, std_acc = cross_validate_subjects(
        X_all[cv_mask], y_all[cv_mask], subjects_all[cv_mask],
        n_splits=3, epochs=30, lr=params["lr"],
        dropout_rate=params["dropout_rate"], f1=params["f1"], f2=f2, d=params["d"],
        chans=64, classes=3, device=device,
    )
    grid_results.append({**params, "f2": f2, "mean_acc": mean_acc, "std_acc": std_acc})

grid_df = pd.DataFrame(grid_results).sort_values("mean_acc", ascending=False)
print(grid_df.to_string(index=False))

## 8. Final Model (64ch) — Retrain Best

In [ ]:
best_params = grid_df.iloc[0].to_dict()
print(f"Best: {best_params}")

final_model = EEGNet(
    chans=64, classes=3, time_points=X_all.shape[2],
    f1=int(best_params["f1"]), f2=int(best_params["f2"]),
    d=int(best_params["d"]), dropout_rate=best_params["dropout_rate"],
).to(device)

final_dl = DataLoader(EEGDataset(X_all[cv_mask], y_all[cv_mask]), batch_size=64, shuffle=True)
final_opt = torch.optim.Adam(final_model.parameters(), lr=best_params["lr"])
final_sched = torch.optim.lr_scheduler.CosineAnnealingLR(final_opt, T_max=50)
w = compute_class_weight("balanced", classes=np.unique(y_all[cv_mask]), y=y_all[cv_mask])
final_loss = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(device))

final_results, _ = train(
    final_model, final_dl, loaders["test_loader"], loaders["test_loader"],
    final_opt, final_loss, final_sched, device, epochs=50,
)
preds, labels = predict_with_model(final_model, loaders["test_loader"], device)
print_evaluation(labels, preds, ["rest", "left_hand", "right_hand"], "Final 64ch")
plot_confusion_matrix(labels, preds, ["rest", "left_hand", "right_hand"])
save_model(final_model, "outputs/eegnet_64ch_3class")

## 9. CSP + ML Grid Search (64ch)

In [ ]:
ml_results = run_csp_ml_grid(
    X_all[cv_mask], y_all[cv_mask], subjects_all[cv_mask],
    task_mode="ternary", n_splits=5,
)

results_df = pd.DataFrame([
    {"Model": r["model"], "CV Accuracy": r["best_cv_acc"], "Time (s)": r["time_s"]}
    for r in ml_results
]).sort_values("CV Accuracy", ascending=False)
print(results_df.to_string(index=False))
plot_model_comparison(results_df, title="CSP+ML 64ch 3-class")

best = max(ml_results, key=lambda x: x["best_cv_acc"])
y_pred = best["grid_obj"].best_estimator_.predict(split["X_test"])
print_evaluation(split["y_test"], y_pred, ["rest", "left_hand", "right_hand"], f'Best: {best["model"]}')
plot_confusion_matrix(split["y_test"], y_pred, ["rest", "left_hand", "right_hand"])

---
## 10. Motor Cortex Channels (21ch) — Re-epoch

In [ ]:
motor_ch = cfg["channels"]["motor_channels"]

X_all, y_all, subjects_all, _ = epoch_subjects(
    raw_data,
    event_id={"rest": 1, "left_hand": 2, "right_hand": 3},
    channels=motor_ch,
    low_freq=7.0, high_freq=30.0, tmin=0.0, tmax=4.0,
    label_offset=1, seed=cfg["seed"],
)
split = subject_split(X_all, y_all, subjects_all, seed=cfg["seed"])
loaders = make_dataloaders(split, batch_size=64)
cv_mask = split["train_mask"] | split["val_mask"]
print(f"21ch shape: {X_all.shape}")

## 11. EEGNet (21ch, 3-class)

In [ ]:
eeg_net = EEGNet(chans=21, classes=3, time_points=X_all.shape[2]).to(device)
w = compute_class_weight("balanced", classes=np.unique(split["y_train"]), y=split["y_train"])
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(device))
opt = torch.optim.Adam(eeg_net.parameters(), lr=0.001)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=55)
results, _ = train(eeg_net, loaders["train_loader"], loaders["val_loader"],
                   loaders["test_loader"], opt, loss_fn, sched, device, epochs=55)
plot_training_curves(results, "EEGNet 21ch 3-class")
preds, labels = predict_with_model(eeg_net, loaders["test_loader"], device)
print_evaluation(labels, preds, ["rest", "left_hand", "right_hand"])
plot_confusion_matrix(labels, preds, ["rest", "left_hand", "right_hand"])

## 12. CSP + ML (21ch, 3-class)

In [ ]:
ml_21 = run_csp_ml_grid(X_all[cv_mask], y_all[cv_mask], subjects_all[cv_mask], task_mode="ternary", n_splits=5)
results_df = pd.DataFrame([{"Model": r["model"], "CV Accuracy": r["best_cv_acc"]} for r in ml_21]).sort_values("CV Accuracy", ascending=False)
print(results_df.to_string(index=False))
plot_model_comparison(results_df, title="CSP+ML 21ch 3-class")

---
## 13. Two-Stage: Mu-Wave Gate + L/R EEGNet

In [ ]:
mu_train = compute_mu_power(split["X_train"])
mu_val = compute_mu_power(split["X_val"])
mu_test = compute_mu_power(split["X_test"])

best_thresh, train_f1, _ = find_best_mu_threshold(mu_train, split["y_train"])
print(f"Best mu threshold: {best_thresh:.4f}, F1: {train_f1:.4f}")

# Train binary L/R on active epochs
lr_mask = split["y_train"] > 0
lr_dl = DataLoader(EEGDataset(split["X_train"][lr_mask], split["y_train"][lr_mask] - 1), batch_size=64, shuffle=True)
lr_vdl = DataLoader(EEGDataset(split["X_val"][split["y_val"] > 0], split["y_val"][split["y_val"] > 0] - 1), batch_size=64)
lr_tdl = DataLoader(EEGDataset(split["X_test"][split["y_test"] > 0], split["y_test"][split["y_test"] > 0] - 1), batch_size=64)

lr_model = EEGNet(chans=21, classes=2, time_points=X_all.shape[2]).to(device)
lr_opt = torch.optim.Adam(lr_model.parameters(), lr=0.001)
lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(lr_opt, T_max=100)
lr_w = compute_class_weight("balanced", classes=np.unique(split["y_train"][lr_mask] - 1), y=split["y_train"][lr_mask] - 1)
lr_loss = nn.CrossEntropyLoss(weight=torch.tensor(lr_w, dtype=torch.float32).to(device))
lr_res, _ = train(lr_model, lr_dl, lr_vdl, lr_tdl, lr_opt, lr_loss, lr_sched, device, epochs=100)
plot_training_curves(lr_res, "L/R EEGNet")

y_pred_2s = two_stage_predict(split["X_test"], mu_test, best_thresh, lr_model, device)
print_evaluation(split["y_test"], y_pred_2s, ["rest", "left_hand", "right_hand"], "Two-Stage")
plot_confusion_matrix(split["y_test"], y_pred_2s, ["rest", "left_hand", "right_hand"], cmap="Greens")

---
## 14. Binary L/R (21ch) — Full Pipeline

In [ ]:
X_all, y_all, subjects_all, _ = epoch_subjects(
    raw_data, event_id={"left_hand": 2, "right_hand": 3}, channels=motor_ch,
    low_freq=7.0, high_freq=30.0, tmin=0.0, tmax=4.0, label_offset=2, seed=cfg["seed"],
)
split = subject_split(X_all, y_all, subjects_all, seed=cfg["seed"])
loaders = make_dataloaders(split, batch_size=64)
cv_mask = split["train_mask"] | split["val_mask"]

eeg_net = EEGNet(chans=21, classes=2, time_points=X_all.shape[2]).to(device)
w = compute_class_weight("balanced", classes=np.unique(split["y_train"]), y=split["y_train"])
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(device))
opt = torch.optim.Adam(eeg_net.parameters(), lr=0.001)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
results, _ = train(eeg_net, loaders["train_loader"], loaders["val_loader"],
                   loaders["test_loader"], opt, loss_fn, sched, device, epochs=100)
plot_training_curves(results, "Binary L/R")
preds, labels = predict_with_model(eeg_net, loaders["test_loader"], device)
print_evaluation(labels, preds, ["left_hand", "right_hand"])
plot_confusion_matrix(labels, preds, ["left_hand", "right_hand"])

## 15. Binary L/R — EEGNet Grid Search + CSP+ML

In [ ]:
eegnet_grid_results = run_eegnet_grid(
    X_all[cv_mask], y_all[cv_mask], subjects_all[cv_mask],
    param_grid=cfg["eegnet_grid"], chans=21, classes=2,
    n_splits=3, epochs_train=30, device=device,
)
grid_df = pd.DataFrame(eegnet_grid_results).sort_values("mean_acc", ascending=False)
print(grid_df.to_string(index=False))

ml_bin = run_csp_ml_grid(X_all[cv_mask], y_all[cv_mask], subjects_all[cv_mask], task_mode="binary", n_splits=5)
ml_df = pd.DataFrame([{"Model": r["model"], "CV Accuracy": r["best_cv_acc"]} for r in ml_bin]).sort_values("CV Accuracy", ascending=False)
print(ml_df.to_string(index=False))
plot_model_comparison(ml_df, title="Binary L/R CSP+ML")

---
## 16. Preprocessing Grid Search

In [ ]:
preproc_df = run_preprocessing_grid(
    raw_data,
    preprocessing_grid=cfg["preprocessing_grid"],
    channels=motor_ch,
    task_mode="binary",
    n_splits=3, epochs=30, chans=21, classes=2,
    seed=cfg["seed"], device=device,
)
print(preproc_df.head(10).to_string(index=False))

## 17. Joint Preprocessing x Model Grid Search

In [ ]:
top_preproc = preproc_df.head(5)

joint_df = run_joint_grid(
    raw_data, top_preproc,
    channels=motor_ch, task_mode="binary",
    chans=21, classes=2,
    n_splits=3, eegnet_epochs=30,
    seed=cfg["seed"], device=device,
)
print(joint_df.head(15).to_string(index=False))

## 18. Final Model — Retrain Best

In [ ]:
best = joint_df.iloc[0]
print(f"Global best: {best['model_name']} with {best['preproc']}")
print(f"CV: {best['mean_acc']:.4f}")

# Retrain on train+val, evaluate on test
# (full pattern in train.py)

---
## 19. FBCSP (Stub)

In [ ]:
from src.pipelines.fbcsp import apply_filter_bank, FBCSP_BANDS

filtered = apply_filter_bank(X_all[:10], fs=160.0)
print(f"Filter bank: {len(filtered)} bands, each shape: {filtered[0].shape}")
print(f"Bands: {FBCSP_BANDS}")